## Exercise 1

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
educx GmbH – Modul 3: Neuronale Netze
Tag 13: Hyperparameter-Optimierung
Niveau: Anfänger
Aufgabe 13.1: Grid Search für Lernrate und Batchgröße

Lernziel: Systematische Hyperparameter-Suche mit Grid Search
Datensatz: MNIST
"""

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from itertools import product

(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 784).astype('float32') / 255.0
X_test  = X_test.reshape(-1, 784).astype('float32') / 255.0

# Kleiner Datensatz für schnelle Suche
X_sub = X_train[:5000]
y_sub = y_train[:5000]

def erstelle_modell(lernrate, einheiten):
    model = keras.Sequential([
        keras.layers.Dense(einheiten, activation='relu', input_shape=(784,)),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(lernrate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Hyperparameter-Gitter
lernraten  = [0.001, 0.01, 0.1]
einheiten_ = [32, 64, 128]

ergebnisse = []
print("=== Grid Search ===")
print(f"{'Lernrate':<12} {'Einheiten':<12} {'Val Acc':<12} {'Epochen'}")
print("-" * 50)

for lr, eu in product(lernraten, einheiten_):
    model = erstelle_modell(lr, eu)
    es = keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
    h = model.fit(X_sub, y_sub, epochs=20, batch_size=64,
                  validation_split=0.2, callbacks=[es], verbose=0)
    val_acc = max(h.history['val_accuracy'])
    n_epochen = len(h.history['loss'])
    ergebnisse.append((lr, eu, val_acc, n_epochen))
    print(f"{lr:<12} {eu:<12} {val_acc:.4f}      {n_epochen}")

# Bestes Ergebnis
beste = max(ergebnisse, key=lambda x: x[2])
print(f"\nBeste Kombination: LR={beste[0]}, Einheiten={beste[1]}")
print(f"Beste Val Accuracy: {beste[2]*100:.2f}%")

# Heatmap der Ergebnisse
acc_matrix = np.array([r[2] for r in ergebnisse]).reshape(len(lernraten), len(einheiten_))

plt.figure(figsize=(8, 6))
plt.imshow(acc_matrix, cmap='viridis', aspect='auto')
plt.colorbar(label='Validation Accuracy')
plt.xticks(range(len(einheiten_)), einheiten_)
plt.yticks(range(len(lernraten)), lernraten)
plt.xlabel('Einheiten')
plt.ylabel('Lernrate')
plt.title('Grid Search Heatmap')

for i in range(len(lernraten)):
    for j in range(len(einheiten_)):
        plt.text(j, i, f'{acc_matrix[i,j]:.3f}', ha='center', va='center',
                 color='white', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('grid_search_heatmap.png', dpi=150)
plt.show()


## Exercise 2

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
educx GmbH – Modul 3: Neuronale Netze
Tag 13: Hyperparameter-Optimierung
Niveau: Anfänger
Aufgabe 13.2: Random Search – Effizienter als Grid Search

Lernziel: Random Search und Vorteile gegenüber Grid Search verstehen
Datensatz: MNIST
"""

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 784).astype('float32') / 255.0
X_sub = X_train[:5000]
y_sub = y_train[:5000]

np.random.seed(42)

def zufaellige_hyperparameter():
    return {
        'lernrate':       10 ** np.random.uniform(-4, -1),
        'einheiten_1':    int(np.random.choice([32, 64, 128, 256])),
        'einheiten_2':    int(np.random.choice([16, 32, 64])),
        'dropout_rate':   np.random.uniform(0.0, 0.5),
        'batch_size':     int(np.random.choice([32, 64, 128])),
    }

def erstelle_und_trainiere(hp):
    model = keras.Sequential([
        keras.layers.Dense(hp['einheiten_1'], activation='relu', input_shape=(784,)),
        keras.layers.Dropout(hp['dropout_rate']),
        keras.layers.Dense(hp['einheiten_2'], activation='relu'),
        keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer=keras.optimizers.Adam(hp['lernrate']),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    
    es = keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
    h = model.fit(X_sub, y_sub, epochs=15, batch_size=hp['batch_size'],
                  validation_split=0.2, callbacks=[es], verbose=0)
    return max(h.history['val_accuracy'])

N_VERSUCHE = 15
print(f"Random Search mit {N_VERSUCHE} Versuchen...")
print(f"{'#':<4} {'LR':<10} {'U1':<6} {'U2':<6} {'DO':<6} {'BS':<6} {'Val Acc'}")
print("-" * 55)

ergebnisse = []
for i in range(N_VERSUCHE):
    hp = zufaellige_hyperparameter()
    acc = erstelle_und_trainiere(hp)
    ergebnisse.append((hp, acc))
    print(f"{i+1:<4} {hp['lernrate']:.4f}   {hp['einheiten_1']:<6} {hp['einheiten_2']:<6} "
          f"{hp['dropout_rate']:.2f}  {hp['batch_size']:<6} {acc:.4f}")

bestes = max(ergebnisse, key=lambda x: x[1])
print(f"\nBestes Ergebnis: {bestes[1]*100:.2f}%")
print(f"Beste Hyperparameter:")
for k, v in bestes[0].items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# Scatter Plot
accs = [r[1] for r in ergebnisse]
lrs  = [r[0]['lernrate'] for r in ergebnisse]

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.scatter(np.log10(lrs), accs, c=accs, cmap='viridis', s=80)
plt.xlabel('log10(Lernrate)')
plt.ylabel('Validation Accuracy')
plt.title('LR vs. Accuracy (Random Search)')
plt.colorbar(label='Accuracy')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(sorted(accs), 'o-', color='#00E6FF')
plt.axhline(max(accs), color='red', linestyle='--', label=f'Max: {max(accs):.3f}')
plt.xlabel('Rang')
plt.ylabel('Validation Accuracy')
plt.title('Ergebnisse sortiert')
plt.legend()
plt.grid(True, alpha=0.3)

plt.suptitle(f'Random Search ({N_VERSUCHE} Versuche)')
plt.tight_layout()
plt.savefig('random_search.png', dpi=150)
plt.show()


## Exercise 3

**Dataset Used:** Synthetic Classification Data (sklearn.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
educx GmbH – Modul 3: Neuronale Netze
Tag 13: Hyperparameter-Optimierung
Niveau: Anfänger
Aufgabe 13.3: Kreuzvalidierung für Modellbewertung

Lernziel: K-Fold Kreuzvalidierung für zuverlässige Evaluation
Datensatz: Synthetische Klassifikationsdaten
"""

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import KFold
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
X, y = make_classification(n_samples=500, n_features=20, random_state=42)
scaler = StandardScaler()
X = scaler.fit_transform(X).astype('float32')

def erstelle_modell():
    model = keras.Sequential([
        keras.layers.Dense(64, activation='relu', input_shape=(20,)),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

fold_accs = []
fold_histories = []

print(f"=== {N_SPLITS}-Fold Kreuzvalidierung ===")
for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_va = X[train_idx], X[val_idx]
    y_tr, y_va = y[train_idx], y[val_idx]
    
    model = erstelle_modell()
    h = model.fit(X_tr, y_tr, epochs=50, batch_size=32,
                  validation_data=(X_va, y_va), verbose=0)
    
    acc = model.evaluate(X_va, y_va, verbose=0)[1]
    fold_accs.append(acc)
    fold_histories.append(h)
    print(f"Fold {fold+1}: Validation Accuracy = {acc*100:.2f}%")

print(f"\nMittelwert: {np.mean(fold_accs)*100:.2f}%")
print(f"Standardabweichung: {np.std(fold_accs)*100:.2f}%")
print(f"95% CI: [{(np.mean(fold_accs)-1.96*np.std(fold_accs))*100:.1f}%, "
      f"{(np.mean(fold_accs)+1.96*np.std(fold_accs))*100:.1f}%]")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i, h in enumerate(fold_histories):
    axes[0].plot(h.history['val_accuracy'], label=f'Fold {i+1}', alpha=0.7)
axes[0].set_title('Validation Accuracy pro Fold')
axes[0].set_xlabel('Epoch')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].bar(range(1, N_SPLITS+1), [a*100 for a in fold_accs],
             color='#00E6FF', edgecolor='black')
axes[1].axhline(np.mean(fold_accs)*100, color='red', linestyle='--',
                 label=f'Mittelwert: {np.mean(fold_accs)*100:.1f}%')
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title(f'{N_SPLITS}-Fold Kreuzvalidierung')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim(50, 100)

plt.tight_layout()
plt.savefig('kreuzvalidierung.png', dpi=150)
plt.show()
